# Script for Processing Facial Action Units Data

This script processes facial action units (AUs) data from files, identifies activities, and saves metrics for different emotional expressions.

## Setup

1. **Imports and Configurations**
   - Import necessary libraries: `sys`, `warnings`, `os`, `pandas as pd`
   - Ignore all warnings to avoid clutter.
   - Set matplotlib inline and seaborn style for plotting.

2. **Add Custom Module Path**
   - Add custom module path to the system path to import `AUFeaturesExtraction`.

3. **Define Activities**
   - Define the list of activities: `['emo', 'tb', 'story']`.

4. **Set Directory Path**
   - Set the directory path containing the data files.

## File Processing

1. **Iterate Over Files**
   - Walk through the directory to list all files.

2. **Process Each File**
   - For each file:
     - Extract subject number from the filename.
     - Identify the activity (emo, tb, story) from the filename.
     - Identify the person type ('S' for sub, 'M' for mar) from the filename.
     - If an activity is identified, process the file; otherwise, skip it.

## Data Processing

1. **Read and Prepare Data**
   - Read the CSV file into a DataFrame.
   - Strip whitespace from column names.
   - Add a low peak at the beginning and end of AU columns for smoothing.

2. **Process Different Emotions**
   - **Smiling**
     - Define relevant AUs (`AU12_r`, `AU06_r`).
     - Prepare and smooth signals.
     - Find high and low peaks.
     - Compute single AU and combo AU peak features.
     - Save metrics to CSV files in appropriate directories.
   - **Sading**
     - Define relevant AUs (`AU01_r`, `AU04_r`, `AU15_r`).
     - Follow the same steps as for smiling.
   - **Angering**
     - Define relevant AUs (`AU04_r`, `AU05_r`, `AU07_r`, `AU23_r`).
     - Follow the same steps as for smiling.
   - **Fearing**
     - Define relevant AUs (`AU01_r`, `AU02_r`, `AU04_r`, `AU05_r`).
     - Follow the same steps as for smiling.

3. **Save DataFrames**
   - Save single AU and combo AU peak metrics to CSV files in the specified directories.

## Summary

This script processes AUs data files to identify and save metrics for different emotional expressions (smiling, sading, angering, fearing). It reads data, processes signals, identifies peaks, and saves the results in structured directories based on activity names.

### Example Output

```plaintext
Subject: 219, Activity: emo, Person: S
Saving SMILING...
Found Simple Events: 10
Found Combo Events: 5
Saved Smiling 219_emo_S.csv
Saving SADING...
Found Simple Events: 8
Found Combo Events: 4
Saved Sading 219_emo_S.csv
Saving ANGERING...
Found Simple Events: 7
Found Combo Events: 3
Saved Angering 219_emo_S.csv
Saving FEARING...
Found Simple Events: 6
Found Combo Events: 2
Saved Fearing 219_emo_S.csv


_______________________________________
_______________________________________

### Detailed Explanation of AU Signal Manipulation and Emotion Feature Computation

#### 1. **Standardization and Normalization**

- **StandardizeScaler(feat)**
  - **Library Used**: `numpy`
  - **Description**: Standardizes the input feature array `feat` by subtracting the mean and dividing by the standard deviation.
  - **Formula**:
    \[
    \text{feat\_scaled} = \frac{\text{feat} - \mu}{\sigma}
    \]
    where \(\mu\) is the mean and \(\sigma\) is the standard deviation of `feat`.

- **NormalizeScaler(feat)**
  - **Library Used**: `numpy`
  - **Description**: Normalizes the input feature array `feat` by subtracting the minimum and dividing by the range (max - min).
  - **Formula**:
    \[
    \text{feat\_scaled} = \frac{\text{feat} - \min(\text{feat})}{\max(\text{feat}) - \min(\text{feat})}
    \]

#### 2. **Signal Preparation and Cleaning**

- **PrepareSignal(df, au_list)**
  - **Library Used**: `pandas`
  - **Description**: Filters out rows where the 'success' column is 0 by setting the corresponding AU values to 0.
  - **Operation**:
    ```python
    au_df = df[au_list].copy()
    au_df.loc[df['success'] == 0] = 0
    ```

- **RemoveSubSpeakingFrames(df, au_list)**
  - **Library Used**: `pandas`
  - **Description**: Filters out rows where the 'sub_talking' column is 0 by setting the corresponding AU values to 0.
  - **Operation**:
    ```python
    au_df = df[au_list].copy()
    au_df.loc[df['sub_talking'] == 0] = 0
    ```

#### 3. **Signal Smoothing**

- **SmoothSignal(df, au, lowpass)**
  - **Library Used**: `scipy.signal`
  - **Description**: Smooths the AU signal using a low-pass Butterworth filter.
  - **Steps**:
    1. **Calculate Sample Rate**: Compute the frame rate based on timestamps.
    2. **Apply Butterworth Filter**: A 3-pole low-pass filter is applied at 0.02x Nyquist frequency.
    ```python
    b, a = scipy.signal.butter(N=3, Wn=0.02, btype='lowpass', analog=False)
    au_smooth = scipy.signal.filtfilt(b, a, df[au], method='pad', padlen=200)
    au_smooth = np.where(au_smooth < 0, 0, au_smooth)
    ```

#### 4. **Peak Extraction**

- **FindHLPeaks(df, au_smooth, height, prominence)**
  - **Library Used**: `scipy.signal`
  - **Description**: Finds high and low peaks in the smoothed AU signal using prominence and height thresholds.
  - **Operations**:
    ```python
    peak_hp, properties_hp = find_peaks(au_smooth, height=height, distance=30, width=30, prominence=prominence)
    peak_lp, properties_lp = find_peaks(-au_smooth, prominence=prominence)
    ```

#### 5. **Feature Computation for Single AU**

- **ComputeSingleAUPeaksFeatures(df, au_df, peak_hp, peak_lp, height)**
  - **Libraries Used**: `numpy`, `pandas`
  - **Description**: Computes features for individual AU signals based on the peaks found.
  - **Features Extracted**:
    - **Intensity (int_avg_combo)**: Mean intensity of the peaks.
    - **Prominence (prominence_combo)**: Maximum prominence of the peaks.
    - **Area (area_combo)**: Sum of the signal values in the peaks.
    - **Duration (duration_combo)**: Duration of the peaks in seconds.
    - **Low Peak Positions (peak_lp_combo)**: Positions of the low peaks.
  - **Operation**:
    ```python
    peaks_dict = {
        'int_avg_combo': peaks_int,
        'prominence_combo': peaks_prom,
        'area_combo': peaks_area,
        'duration_combo': peaks_dur,
        'peak_lp_combo': peaks_lp
    }
    peaks_df = pd.DataFrame(peaks_dict)
    ```

#### 6. **Feature Computation for Combined AUs**

- **compute_peaks_for_au(aus_dict, fr, height, start, stop)**
  - **Library Used**: `pandas`
  - **Description**: Computes features for combined AU signals within a specified range (start to stop).
  - **Operation**:
    ```python
    combo_features = {}
    for au_name, au_df in aus_dict.items():
        current_peak = au_df.iloc[start:stop]
        current_peak = current_peak[current_peak > height].dropna()
        if not current_peak.empty:
            peak_prom = float(np.max(current_peak))
            peak_int = float(np.mean(current_peak))
            peak_area = float(np.sum(current_peak))
            peak_dur = float(len(current_peak) / fr)
            combo_features[au_name] = {
                'peak_prom': peak_prom,
                'peak_int': peak_int,
                'peak_area': peak_area,
                'peak_dur': peak_dur
            }
    return combo_features
    ```

- **ComputeComboAUPeaksFeatures(df, aus_dict, peak_hp_dict, peak_lp_dict, height)**
  - **Libraries Used**: `numpy`, `pandas`
  - **Description**: Computes features for combinations of AU signals by considering peaks across multiple AUs.
  - **Features Extracted**: Similar to individual AU features but for combined AU signals.
  - **Operation**:
    ```python
    peaks_combo_dict = {
        'int_avg_combo': peaks_int_combo,
        'prominence_combo': peaks_prom_combo,
        'area_combo': peaks_area_combo,
        'duration_combo': peaks_dur_combo,
        'peak_lp_combo': peaks_lp_combo
    }
    peaks_combo_df = pd.DataFrame(peaks_combo_dict)
    return peaks_combo_df, au_bin_combo
    ```

- **ComputeAverageComboAUPeaksFeatures(df, aus_dict, peak_hp_dict, peak_lp_dict, height)**
  - **Libraries Used**: `numpy`, `pandas`
  - **Description**: Computes average features for combinations of AU signals.
  - **Features Extracted**: Similar to individual and combo AU features, but averaged across all AUs.
  - **Operation**:
    ```python
    peaks_combo_dict = {
        'int_avg_combo': peaks_int_combo,
        'prominence_combo': peaks_prom_combo,
        'area_combo': peaks_area_combo,
        'duration_combo': peaks_dur_combo,
        'peak_lp_combo': peaks_lp_combo
    }
    peaks_combo_df = pd.DataFrame(peaks_combo_dict)
    return peaks_combo_df, au_bin_combo, peaks_combo_df, au_bin_combo
    ```

### Emotion-Specific Processing

- **Expressions Considered**: The script processes different emotions by extracting Action Units (AUs) that correspond to various expressions such as smiles (both combo and simple), sadness, anger, and fear.
  - **Smiles (combo and simple)**: Processes AU12 and AU6.
  - **Sadness**: Processes AU1 and AU4.
  - **Anger**: Processes AU4, AU5, and AU7.
  - **Fear**: Processes AU1, AU2, and AU4.

### Summary of Features Extracted
- **Intensity (int_avg)**: Average intensity of the AU signal within peaks.
- **Prominence (prominence)**: Maximum prominence of the AU signal within peaks.
- **Area (area)**: Sum of AU signal values within peaks.
- **Duration (duration)**: Duration of the AU signal peaks.
- **Low Peak Positions (peak_lp)**: Positions of the low peaks in the AU signal.


### Difference Between `ComputeComboAUPeaksFeatures` and `ComputeAverageComboAUPeaksFeatures`

- **ComputeComboAUPeaksFeatures**
  - **Purpose**: Computes features for combinations of AU signals by considering peaks across multiple AUs.
  - **Method**:
    - For each peak range, it computes the features (intensity, prominence, area, and duration) for each individual AU and stores them separately.
    - The features of each AU are considered individually.
  - **Output**:
    - A DataFrame containing individual features for each AU in the combination.
    - A binary DataFrame indicating the presence of each AU signal within the specified range.

- **ComputeAverageComboAUPeaksFeatures**
  - **Purpose**: Computes average features for combinations of AU signals.
  - **Method**:
    - For each peak range, it computes the features (intensity, prominence, area, and duration) for each individual AU.
    - It then averages these features across all AUs in the combination.
    - The result is a single set of features representing the average characteristics of the combined AUs.
  - **Output**:
    - A DataFrame containing the averaged features for the combination of AUs.
    - A binary DataFrame indicating the presence of the combined AU signals within the specified range.
    - Note: In the provided script, the function was modified to return the same DataFrame twice. Ensure to update it to match your actual requirements.


--------------------------------------------------------------


In [ ]:
import sys
import warnings
sys.path.insert(0, '/home/galvari/BAYAN/src')
from AUFeaturesExtraction_backup import *
%matplotlib inline
sns.set_style("darkgrid")
# Ignore all warnings
warnings.filterwarnings("ignore")




activities = ['emo', 'tb', 'story']

dir_path = '/home/galvari/BAYAN/DATA/ShareWithGian/openface_gian/'
pd.options.mode.chained_assignment = None

for root, dirs, files in os.walk(dir_path):
    for filename in files:
        file_path = os.path.join(root, filename)
        if os.path.isfile(file_path):
            parts = filename.split('_')
            subject_number = parts[0]
            activity_name = None
            person = 'S'  # Default to 'S'



            for part in parts:
                lower_part = part.lower()
                for activity in activities:
                    if activity in lower_part:
                        activity_name = activity
                        break  # No need to check further once activity is found

                if 'sub' in lower_part:
                    person = 'S'
                elif 'mar' in lower_part:
                    person = 'I'
                elif 'cropped' in lower_part:
                    subject_number = '1000' + subject_number

            if activity_name:
                print('-----------------------------------------------------------------------')
                print(f"Subject: {subject_number}, Activity: {activity_name}, Person: {person}")
                ##### PREPARE SIGNAL ######
                df = pd.read_csv(file_path)
                df.columns = df.columns.str.strip()
                timestamp = df['timestamp'].iloc[-1]

                frame_rate = calculate_frame_rate(df)
                print('Frame Rate: ', frame_rate)

                file_name_save = subject_number+'_'+activity_name+'_'+person+'.csv'


                print('Saving SMILING...')
                AU1 = 'AU12_r'
                AU2 = 'AU06_r'
                AUs = ['AU12_r','AU06_r']
                # add a last element to have a low pyke at beggining and at end
                df[AU1].iloc[0] = 5
                df[AU1].iloc[-1] = 5
                df[AU2].iloc[0] = 5
                df[AU2].iloc[-1] = 5
                # filtering by confidence
                df_succ = PrepareSignal(df, AUs)
                # smooth aus signals
                au12_smooth = SmoothSignal(df_succ, au=AU1, lowpass=0.02, fr = frame_rate)
                au6_smooth = SmoothSignal(df_succ, au=AU2, lowpass=0.02, fr = frame_rate)
                # get peaks lp/hp for aus
                peak_hp_12, properties_hp_12, peak_lp_12, properties_lp_12 = FindHLPeaks(df_succ, au12_smooth, height=1.0, prominence=0.3)
                peak_hp_6, properties_hp_6, peak_lp_6, properties_lp_6 = FindHLPeaks(df_succ, au6_smooth, height=1.0, prominence=0.3)
                # create df for aus
                au12_smooth_df = pd.DataFrame(au12_smooth)
                au6_smooth_df = pd.DataFrame(au6_smooth)
                # save peaks metrics for single aus
                au12_peaks, au12_bin = ComputeSingleAUPeaksFeatures(df_succ, au12_smooth_df, peak_hp_12, peak_lp_12, height = 0.9, fr = frame_rate)
                au6_peaks, au6_bin = ComputeSingleAUPeaksFeatures(df_succ, au6_smooth_df, peak_hp_6, peak_lp_6, height = 0.9, fr = frame_rate)
                # save peaks metrics for SMILE COMBO aus
                aus_dict = {'AU12':au12_smooth_df,'AU6': au6_smooth_df}
                peak_hp_dict = {'peak_hp_12':peak_hp_12,'peak_hp_6':peak_hp_6}
                peak_lp_dict = {'peak_lp_12':peak_lp_12,'peak_lp_6':peak_lp_6}
                au12simple_peaks, au12simple_bin, au12combo_peaks, au12combo_bin = ComputeAverageComboAUPeaksFeatures(df_succ, aus_dict, peak_hp_dict, peak_lp_dict,height=0.9, fr = frame_rate)
                au12combo_peaks['timestamp'] = timestamp
                au12simple_peaks['timestamp'] = timestamp
                combo_path = '/home/galvari/BAYAN/DATA/FeaturesAU/'+activity_name+'/ComboSmiling/'
                simple_path = '/home/galvari/BAYAN/DATA/FeaturesAU/'+activity_name+'/SimpleSmiling/'
                if not os.path.exists(combo_path):
                    os.makedirs(combo_path)
                if not os.path.exists(simple_path):
                    os.makedirs(simple_path)
                #save dataframes
                au12simple_peaks.to_csv(simple_path+file_name_save)
                au12combo_peaks.to_csv(combo_path+file_name_save)
                print('Found Simple Events: ', len(au12simple_peaks))
                print('Found Combo Events: ', len(au12combo_peaks))

                if len(au12combo_peaks)>0:
                    print('First Event dur: ', au12combo_peaks.duration_combo.iloc[0])
                    print('First Event frames: ', au12combo_peaks.peak_lp_combo.iloc[0])
                print('Saved Smiling ', file_name_save)


                print('Saving SADING...')
                AU1 = 'AU01_r'
                AU2 = 'AU04_r'
                AU3 = 'AU15_r'
                AUs = ['AU01_r','AU04_r','AU15_r']
                # add a last element to have a low pyke at beggining and at end
                df[AU1].iloc[0] = 5
                df[AU1].iloc[-1] = 5
                df[AU2].iloc[0] = 5
                df[AU2].iloc[-1] = 5
                df[AU3].iloc[0] = 5
                df[AU3].iloc[-1] = 5
                # filtering by confidence
                df_succ = PrepareSignal(df, AUs)
                # smooth aus signals
                au1_smooth = SmoothSignal(df_succ, au=AU1, lowpass=0.02, fr = frame_rate)
                au4_smooth = SmoothSignal(df_succ, au=AU2, lowpass=0.02, fr = frame_rate)
                au15_smooth = SmoothSignal(df_succ, au=AU3, lowpass=0.02, fr = frame_rate)
                # get peaks lp/hp for aus
                peak_hp_1, properties_hp_1, peak_lp_1, properties_lp_1 = FindHLPeaks(df_succ, au1_smooth, height=1.0, prominence=0.3)
                peak_hp_4, properties_hp_4, peak_lp_4, properties_lp_4 = FindHLPeaks(df_succ, au4_smooth, height=1.0, prominence=0.3)
                peak_hp_15, properties_hp_15, peak_lp_15, properties_lp_15 = FindHLPeaks(df_succ, au15_smooth, height=1.0, prominence=0.3)
                # create df for aus
                au1_smooth_df = pd.DataFrame(au1_smooth)
                au4_smooth_df = pd.DataFrame(au4_smooth)
                au15_smooth_df = pd.DataFrame(au15_smooth)
                # save peaks metrics for single aus
                au1_peaks, au1_bin = ComputeSingleAUPeaksFeatures(df_succ, au1_smooth_df, peak_hp_1, peak_lp_1,height=0.9, fr = frame_rate)
                au4_peaks, au4_bin = ComputeSingleAUPeaksFeatures(df_succ, au4_smooth_df, peak_hp_4, peak_lp_4,height=0.9, fr = frame_rate)
                au15_peaks, au15_bin = ComputeSingleAUPeaksFeatures(df_succ, au15_smooth_df, peak_hp_15, peak_lp_15,height=0.9, fr = frame_rate)
                # save peaks metrics for SMILE COMBO aus
                aus_dict = {'AU1':au1_smooth_df,'AU4': au4_smooth_df}
                peak_hp_dict = {'peak_hp_1':peak_hp_1,'peak_hp_4':peak_hp_4}
                peak_lp_dict = {'peak_lp_1':peak_lp_1,'peak_lp_4':peak_lp_4}
                au1simple_peaks, au1simple_bin, au1combo_peaks, au1combo_bin = ComputeAverageComboAUPeaksFeatures(df_succ, aus_dict, peak_hp_dict, peak_lp_dict,height=0.9, fr = frame_rate)
                au1combo_peaks['timestamp'] = timestamp
                au1simple_peaks['timestamp'] = timestamp
                combo_path = '/home/galvari/BAYAN/DATA/FeaturesAU/'+activity_name+'/ComboSading/'
                simple_path = '/home/galvari/BAYAN/DATA/FeaturesAU/'+activity_name+'/SimpleSading/'
                if not os.path.exists(combo_path):
                    os.makedirs(combo_path)
                if not os.path.exists(simple_path):
                    os.makedirs(simple_path)
                #save dataframes
                au1simple_peaks.to_csv(simple_path+file_name_save)
                au1combo_peaks.to_csv(combo_path+file_name_save)
                print('Found Simple Events: ', len(au1simple_peaks))
                print('Found Combo Events: ', len(au1combo_peaks))


                print('Saving ANGERING...')
                AU1 = 'AU04_r'
                AU2 = 'AU05_r'
                AU3 = 'AU07_r'
                AU4 = 'AU23_r'
                AUs = ['AU04_r','AU05_r','AU07_r','AU23_r']
                # add a last element to have a low pyke at beggining and at end
                df[AU1].iloc[0] = 5
                df[AU1].iloc[-1] = 5
                df[AU2].iloc[0] = 5
                df[AU2].iloc[-1] = 5
                df[AU3].iloc[0] = 5
                df[AU3].iloc[-1] = 5
                df[AU4].iloc[0] = 5
                df[AU4].iloc[-1] = 5
                # filtering by confidence
                df_succ = PrepareSignal(df, AUs)
                # smooth aus signals
                au4_smooth = SmoothSignal(df_succ, au=AU1, lowpass=0.02, fr = frame_rate)
                au5_smooth = SmoothSignal(df_succ, au=AU2, lowpass=0.02, fr = frame_rate)
                au7_smooth = SmoothSignal(df_succ, au=AU3, lowpass=0.02, fr = frame_rate)
                au23_smooth = SmoothSignal(df_succ, au=AU4, lowpass=0.02, fr = frame_rate)
                # get peaks lp/hp for aus
                peak_hp_4, properties_hp_4, peak_lp_4, properties_lp_4 = FindHLPeaks(df_succ, au4_smooth, height=1.0, prominence=0.3)
                peak_hp_5, properties_hp_5, peak_lp_5, properties_lp_5 = FindHLPeaks(df_succ, au5_smooth, height=1.0, prominence=0.3)
                peak_hp_7, properties_hp_7, peak_lp_7, properties_lp_7 = FindHLPeaks(df_succ, au7_smooth, height=1.0, prominence=0.3)
                peak_hp_23, properties_hp_23, peak_lp_23, properties_lp_23 = FindHLPeaks(df_succ, au23_smooth, height=1.0, prominence=0.3)
                # create df for aus
                au4_smooth_df = pd.DataFrame(au4_smooth)
                au5_smooth_df = pd.DataFrame(au5_smooth)
                au7_smooth_df = pd.DataFrame(au7_smooth)
                au23_smooth_df = pd.DataFrame(au23_smooth)
                # save peaks metrics for single aus
                au4_peaks, au4_bin = ComputeSingleAUPeaksFeatures(df_succ, au4_smooth_df, peak_hp_4, peak_lp_4,height=0.9, fr = frame_rate)
                au5_peaks, au5_bin = ComputeSingleAUPeaksFeatures(df_succ, au5_smooth_df, peak_hp_5, peak_lp_5,height=0.9, fr = frame_rate)
                au7_peaks, au7_bin = ComputeSingleAUPeaksFeatures(df_succ, au7_smooth_df, peak_hp_7, peak_lp_7,height=0.9, fr = frame_rate)
                au23_peaks, au23_bin = ComputeSingleAUPeaksFeatures(df_succ, au23_smooth_df, peak_hp_23, peak_lp_23,height=0.9, fr = frame_rate)
                # save peaks metrics for ANGER COMBO aus
                aus_dict = {'AU4':au4_smooth_df,'AU5': au5_smooth_df,'AU7': au7_smooth_df}
                peak_hp_dict = {'peak_hp_4':peak_hp_4,'peak_hp_5':peak_hp_5,'peak_hp_7':peak_hp_7}
                peak_lp_dict = {'peak_lp_4':peak_lp_4,'peak_lp_5':peak_lp_5,'peak_lp_7':peak_lp_7}
                au4simple_peaks, au4simple_bin, au4combo_peaks, au4combo_bin = ComputeAverageComboAUPeaksFeatures(df_succ, aus_dict, peak_hp_dict, peak_lp_dict,height=0.9, fr = frame_rate)
                au4combo_peaks['timestamp'] = timestamp
                au4simple_peaks['timestamp'] = timestamp
                combo_path = '/home/galvari/BAYAN/DATA/FeaturesAU/'+activity_name+'/ComboAngering/'
                simple_path = '/home/galvari/BAYAN/DATA/FeaturesAU/'+activity_name+'/SimpleAngering/'
                if not os.path.exists(combo_path):
                    os.makedirs(combo_path)
                if not os.path.exists(simple_path):
                    os.makedirs(simple_path)
                #save dataframes
                au4simple_peaks.to_csv(simple_path+file_name_save)
                au4combo_peaks.to_csv(combo_path+file_name_save)
                print('Found Simple Events: ', len(au4simple_peaks))
                print('Found Combo Events: ', len(au4combo_peaks))


                print('Saving FEARING...')
                AU1 = 'AU01_r'
                AU2 = 'AU02_r'
                AU3 = 'AU04_r'
                AU4 = 'AU05_r'
                AUs = ['AU01_r','AU02_r','AU04_r','AU05_r']
                # add a last element to have a low pyke at beggining and at end
                df[AU1].iloc[0] = 5
                df[AU1].iloc[-1] = 5
                df[AU2].iloc[0] = 5
                df[AU2].iloc[-1] = 5
                df[AU3].iloc[0] = 5
                df[AU3].iloc[-1] = 5
                df[AU4].iloc[0] = 5
                df[AU4].iloc[-1] = 5
                # filtering by confidence
                df_succ = PrepareSignal(df, AUs)
                # smooth aus signals
                au1_smooth = SmoothSignal(df_succ, au=AU1, lowpass=0.02, fr = frame_rate)
                au2_smooth = SmoothSignal(df_succ, au=AU2, lowpass=0.02, fr = frame_rate)
                au4_smooth = SmoothSignal(df_succ, au=AU3, lowpass=0.02, fr = frame_rate)
                au5_smooth = SmoothSignal(df_succ, au=AU4, lowpass=0.02, fr = frame_rate)
                # get peaks lp/hp for aus
                peak_hp_1, properties_hp_1, peak_lp_1, properties_lp_1 = FindHLPeaks(df_succ, au1_smooth, height=1.0, prominence=0.3)
                peak_hp_2, properties_hp_2, peak_lp_2, properties_lp_2 = FindHLPeaks(df_succ, au2_smooth, height=1.0, prominence=0.3)
                peak_hp_4, properties_hp_4, peak_lp_4, properties_lp_4 = FindHLPeaks(df_succ, au4_smooth, height=1.0, prominence=0.3)
                peak_hp_5, properties_hp_5, peak_lp_5, properties_lp_5 = FindHLPeaks(df_succ, au5_smooth, height=1.0, prominence=0.3)
                # create df for aus
                au1_smooth_df = pd.DataFrame(au1_smooth)
                au2_smooth_df = pd.DataFrame(au2_smooth)
                au4_smooth_df = pd.DataFrame(au4_smooth)
                au5_smooth_df = pd.DataFrame(au5_smooth)
                # save peaks metrics for single aus
                au1_peaks, au1_bin = ComputeSingleAUPeaksFeatures(df_succ, au1_smooth_df, peak_hp_1, peak_lp_1,height=0.9, fr = frame_rate)
                au2_peaks, au2_bin = ComputeSingleAUPeaksFeatures(df_succ, au2_smooth_df, peak_hp_2, peak_lp_2,height=0.9, fr = frame_rate)
                au4_peaks, au4_bin = ComputeSingleAUPeaksFeatures(df_succ, au4_smooth_df, peak_hp_4, peak_lp_4,height=0.9, fr = frame_rate)
                au5_peaks, au5_bin = ComputeSingleAUPeaksFeatures(df_succ, au5_smooth_df, peak_hp_5, peak_lp_5,height=0.9, fr = frame_rate)
                # save peaks metrics for FEAR COMBO aus
                aus_dict = {'AU1':au1_smooth_df,'AU2': au2_smooth_df,'AU4': au4_smooth_df}
                peak_hp_dict = {'peak_hp_1':peak_hp_1,'peak_hp_2':peak_hp_2,'peak_hp_4':peak_hp_4}
                peak_lp_dict = {'peak_lp_1':peak_lp_1,'peak_lp_2':peak_lp_2,'peak_lp_4':peak_lp_4}
                au1simple_peaks, au1simple_bin, au1combo_peaks, au1combo_bin = ComputeAverageComboAUPeaksFeatures(df_succ, aus_dict, peak_hp_dict, peak_lp_dict,height=0.9, fr = frame_rate)
                au1combo_peaks['timestamp']= timestamp
                au1simple_peaks['timestamp'] = timestamp
                combo_path = '/home/galvari/BAYAN/DATA/FeaturesAU/'+activity_name+'/ComboFearing/'
                simple_path = '/home/galvari/BAYAN/DATA/FeaturesAU/'+activity_name+'/SimpleFearing/'
                if not os.path.exists(combo_path):
                    os.makedirs(combo_path)
                if not os.path.exists(simple_path):
                    os.makedirs(simple_path)
                #save dataframes
                au1simple_peaks.to_csv(simple_path+file_name_save)
                au1combo_peaks.to_csv(combo_path+file_name_save)
                print('Found Simple Events: ', len(au1simple_peaks))
                print('Found Combo Events: ', len(au1combo_peaks))


            else:
                print(f"Skipped file: {filename}")



-----------------------------------------------------------------------
Subject: 1000201, Activity: emo, Person: S
Frame Rate:  30.003037117254202
Saving SMILING...
Number of given AUs:  2
Found Simple Events:  7
Found Combo Events:  4
First Event dur:  3.949600153374233
First Event frames:  (3354, 3755)
Saved Smiling  1000201_emo_S.csv
Saving SADING...
Number of given AUs:  2
Found Simple Events:  2
Found Combo Events:  3
Saving ANGERING...
Number of given AUs:  3
Found Simple Events:  0
Found Combo Events:  1
Saving FEARING...
Number of given AUs:  3
Found Simple Events:  1
Found Combo Events:  0
-----------------------------------------------------------------------
Subject: 201, Activity: tb, Person: S
Frame Rate:  30.019424333392195
Saving SMILING...
Number of given AUs:  2
Found Simple Events:  1
Found Combo Events:  1
First Event dur:  3.0813382352941177
First Event frames:  (572, 1210)
Saved Smiling  201_tb_S.csv
Saving SADING...
Number of given AUs:  2
Found Simple Events:  0
